# ex005_PHT3D_05

In [ ]:
import pandas as pd
from IPython.display import display

comparison_rows = []
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CASE_DIR = Path.cwd()
output = CASE_DIR / "output"
results = np.load(output / "results.npy")
headings = (output / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
times = np.load(output / "results_times.npy")
CASE_DIR = Path.cwd()
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = output
result_times = times
from matplotlib.ticker import LogLocator, MultipleLocator, NullFormatter

REFERENCE_FILE = INPUT_DIR / "PHT3D_05_results.npy"
results = np.load(OUTPUT_DIR / "results.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
result_times = np.load(OUTPUT_DIR / "results_times.npy")
component = {name.strip(): index for index, name in enumerate(headings)}
INJECTION_RATE_M3_PER_DAY = 21.0 * 24.0
OBSERVATION_CELL = -1
mf6pqc_time = result_times
mf6pqc_volume = mf6pqc_time * INJECTION_RATE_M3_PER_DAY
reference = np.load(REFERENCE_FILE)
pht3d_time = reference["time_days"]
pht3d_volume = pht3d_time * INJECTION_RATE_M3_PER_DAY
pht3d = {name: reference[name] for name in ("Ca", "Cl", "Na", "Mg")}
observations = np.load(INPUT_DIR / "observations.npy", allow_pickle=False)
observed_volume = observations["volume_m3"]
observed = {name: observations[name] for name in ("Cl", "Ca", "Mg")}
for name in ("Ca", "Mg", "Na", "Cl"):
    difference = (
        np.interp(pht3d_time, result_times, results[:, component[name], -1]) - reference[name]
    )
    comparison_rows.append(
        {"Variable": name, "RMSE": np.sqrt(np.mean(difference**2)), "Unit": "mol/L"}
    )
colors = {"Mg": "#1f5cff", "Ca": "#e6392f", "Na": "#15803d", "Cl": "#1f5cff"}
markers = {"Mg": "d", "Ca": "^", "Cl": "o"}
style = {
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.linewidth": 0.85,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 9,
}
with plt.rc_context(style):
    fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(9.0, 7.4), gridspec_kw={"hspace": 0.2})
    positive_volume = mf6pqc_volume > 0
    for species in ("Mg", "Ca", "Na"):
        ax_top.plot(
            mf6pqc_volume[positive_volume],
            results[positive_volume, component[species], OBSERVATION_CELL],
            color=colors[species],
            lw=1.8,
            label=f"{species} (MF6PQC)",
        )
    for species in ("Mg", "Ca", "Na"):
        ax_top.plot(
            pht3d_volume[1:],
            pht3d[species][1:],
            color=colors[species],
            lw=1.15,
            ls="-.",
            label=f"{species} (PHT3D)",
        )
    for species in ("Mg", "Ca"):
        ax_top.plot(
            observed_volume,
            observed[species],
            ls="none",
            marker=markers[species],
            ms=4.2,
            mfc=colors[species],
            mec="#111827",
            mew=0.45,
            label=f"{species} (observed)",
            zorder=4,
        )
    ax_top.set(xscale="log", yscale="log", xlim=(100.0, 100000.0), ylim=(0.0001, 1.0))
    ax_top.set_title("Observation well S23", pad=4)
    ax_top.set_ylabel("Concentration (mol L$^{-1}$)")
    ax_top.xaxis.set_major_locator(LogLocator(base=10))
    ax_top.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax_top.xaxis.set_minor_formatter(NullFormatter())
    ax_top.yaxis.set_major_locator(LogLocator(base=10))
    ax_top.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax_top.yaxis.set_minor_formatter(NullFormatter())
    ax_top.legend(
        loc="upper right",
        ncol=1,
        frameon=True,
        fancybox=False,
        framealpha=0.96,
        edgecolor="#64748b",
        columnspacing=1.1,
        handlelength=2.4,
        borderpad=0.55,
    )
    ax_bottom.plot(
        mf6pqc_volume,
        results[:, component["Cl"], OBSERVATION_CELL],
        color=colors["Cl"],
        lw=1.8,
        label="Cl (MF6PQC)",
    )
    ax_bottom.plot(
        pht3d_volume, pht3d["Cl"], color=colors["Cl"], lw=1.15, ls="-.", label="Cl (PHT3D)"
    )
    ax_bottom.plot(
        observed_volume,
        observed["Cl"],
        ls="none",
        marker=markers["Cl"],
        ms=4.2,
        mfc=colors["Cl"],
        mec="#111827",
        mew=0.45,
        label="Cl (observed)",
        zorder=4,
    )
    ax_bottom.set(xlim=(0, 1200), ylim=(0, 0.17))
    ax_bottom.set_xlabel("Volume injected (m$^3$)")
    ax_bottom.set_ylabel("Concentration (mol L$^{-1}$)")
    ax_bottom.xaxis.set_major_locator(MultipleLocator(200))
    ax_bottom.xaxis.set_minor_locator(MultipleLocator(100))
    ax_bottom.yaxis.set_major_locator(MultipleLocator(0.02))
    ax_bottom.legend(
        loc="upper right",
        frameon=True,
        fancybox=False,
        framealpha=0.96,
        edgecolor="#64748b",
        handlelength=2.4,
        borderpad=0.55,
    )
    for ax in (ax_top, ax_bottom):
        ax.grid(which="major", color="#cbd5e1", lw=0.55, alpha=0.42)
        ax.tick_params(which="major", length=4.0, width=0.75)
        ax.tick_params(which="minor", length=2.2, width=0.55)
        for spine in ax.spines.values():
            spine.set_color("#334155")
    fig.align_ylabels((ax_top, ax_bottom))
    plt.show()
comparison = pd.DataFrame(comparison_rows)
comparison = comparison.set_index("Variable")
display(comparison.style.format({"RMSE": "{:.6g}"}).set_uuid("ex005_1"))